# W04 tutorial — the Benchmark Kit

**ACFI20009 Financial Data — 26/27**  
Dr Zeming Li, University of Bristol

**This week you build the Benchmark Kit:** the scoreboard for any set of permnos — equal-weighted, value-weighted with properly lagged caps, and mean-variance — compared gross and net of costs on one panel, with `ann_stats` and `ann_sharpe` as the reusable pieces.

Work straight down the notebook. Each section explains what we are doing and why, then does it. By the last cell the artefact exists and runs.

> ### 🤖 Working with Gemini in this notebook
>
> Colab has Gemini built in. You are **encouraged** to use it — reading and
> directing AI-written code is part of what this unit teaches.
>
> **One hard rule:** never paste rows of CRSP data into Gemini (or any other
> AI tool). The licence forbids it. Describe the *schema* instead — column
> names, dtypes, what a row means — which is all an assistant needs to write
> correct code.

## Setup

Run these two cells once, top to bottom, before anything else.

In [ ]:
# Nothing to install: this notebook uses only what Colab already ships
# (pandas, numpy, matplotlib, statsmodels, scipy, sklearn, pyarrow).

In [ ]:
# --- Data -------------------------------------------------------------------
# The unit's datasets are distributed through Blackboard. Download the
# `ACFI20009_data` folder once, put it in your own Google Drive, and this cell
# will find it. Nothing licensed is ever stored in the public repository, and
# the data never leaves your personal Drive.
from pathlib import Path

try:                                    # running on Colab
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/ACFI20009/data")
except ModuleNotFoundError:             # running locally (convenor)
    DATA = Path("2627/data")

assert DATA.exists(), (
    f"Data folder not found at {DATA}.\n"
    "Download it from Blackboard and place it at MyDrive/ACFI20009/data"
)
print("Data:", DATA, "|", len(list(DATA.glob('*.parquet'))), "parquet files")

---

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
pd.options.display.float_format = "{:.4f}".format

In [ ]:
crsp = pd.read_parquet(DATA / "crsp_monthly.parquet")
crsp = crsp.sort_values(["permno", "date"])

SEED_PERMNOS = [14593, 10107, 47896]     # <- replace with YOUR three permnos
START, END = "2010-01-01", "2019-12-31"  # 120 months

### 1. Expand each permno to its industry

A serious strategy works with an industry universe, not just three stocks. We take each seed permno's **two-digit SIC code** (the broad industry group — `siccd // 100`) and admit every stock sharing it:

In [ ]:
window = crsp[(crsp["date"] >= START) & (crsp["date"] <= END)].copy()
window["sic2"] = window["siccd"] // 100

seed_sic2 = sorted(window.loc[window["permno"].isin(SEED_PERMNOS)]
                   .groupby("permno")["sic2"].last().unique().tolist())
print(f"industry groups: {seed_sic2}")

universe = window[window["sic2"].isin(seed_sic2)].copy()
print(f"stocks before history filter: {universe['permno'].nunique()}")

For the example triple the groups are `[35, 60, 73]` — industrial machinery and computers, banking, and business services.

### 2. Keep complete histories, build the date-by-permno return table

For today's matrix work we need a balanced panel — every stock, every month — arranged with dates down the side and one column per permno:

In [ ]:
n_months = universe["date"].nunique()
counts = universe.groupby("permno")["ret"].count()
full = counts[counts == n_months].index
universe = universe[universe["permno"].isin(full)].copy()

rets = universe.pivot(index="date", columns="permno", values="ret")
assert rets.notna().all().all()          # balanced: no gaps anywhere
print(f"universe: {rets.shape[1]} stocks x {rets.shape[0]} months")

For the example triple that leaves 452 stocks × 120 months.

#### ⚠️ We just committed survivorship bias — on purpose
Requiring a complete 2010–2019 history silently deletes every stock that delisted in the window — the Week 01 villain, and exactly the bias the Week 03 delisting case study quantified. That is acceptable *here*, because we need a balanced matrix to demonstrate a numerical failure, and we are saying so out loud. It is **not** acceptable in a backtest anyone is meant to believe, where the universe must include the stocks that die. Any performance number computed on today's universe is upward-biased; treat it as a laboratory, not a result.

### 3. Two helpers for the rest of the session

The lecture's MVO function and an annualiser (monthly data: mean × 12, volatility × $\sqrt{12}$):

In [ ]:
def mvo_weights(rets, gamma=10.0):
    mu = rets.mean().to_numpy()
    Sigma = rets.cov().to_numpy()
    ones = np.ones(len(mu))
    Sinv_mu = np.linalg.solve(Sigma, mu)    # solve, don't invert
    Sinv_1 = np.linalg.solve(Sigma, ones)
    lam = (ones @ Sinv_mu - gamma) / (ones @ Sinv_1)
    w = (Sinv_mu - lam * Sinv_1) / gamma
    assert np.isclose(w.sum(), 1.0)         # Week 02 habit: assert before you trust
    return w

def ann_stats(x, name):
    return pd.Series({"ann_mean": 12 * x.mean(),
                      "ann_vol": np.sqrt(12) * x.std(),
                      "ann_sharpe": np.sqrt(12) * x.mean() / x.std()}, name=name)

That is everything the rest of the notebook needs. Asking an AI assistant to explain or extend any of the code from here on is fine — but verify what it hands you against the data, for the reason section 9 makes concrete.

### 4. The one-line benchmark


The equal-weighted (1/N) portfolio is the benchmark everything else has to beat, so it goes first: its monthly return series, annualised mean, volatility and Sharpe ratio, and the growth of £1 over the window.

The EW return is the cross-sectional mean each month. We can group the stock-month rows of `universe` by date, or take the mean across each row of `rets`:


In [ ]:
ew = universe.groupby("date")["ret"].mean()      # group the stock-month rows by date
assert np.allclose(ew, rets.mean(axis=1))        # row means of rets agree

print(ann_stats(ew, "EW"))
print(f"growth of £1: £{(1 + ew).prod():.2f}")

For the example universe the annualised mean is about 15.5%, the volatility about 15.5%, the Sharpe ratio close to 1.0, and £1 grows to roughly £4.16. Your seed will differ, but for any US equity universe over 2010–2019 (a strong bull market, survivorship-boosted — see the survivorship warning at the end of section 2) expect a healthy positive Sharpe. No estimation, no optimisation: this is the strategy with nothing to go wrong.

### 5. Value weights, lagged properly

Value weighting needs the same statistics again, but with weights built from *lagged* market caps — the caps at the end of month $t-1$, which is when the portfolio was actually formed. Two things can go wrong in this block, and both are worth holding in mind as you read it: using same-month `mktcap`, and writing `universe["mktcap"].shift(1)` without the `groupby`. The code below avoids both; what each would have cost us comes straight after.


In [ ]:
universe["mktcap_lag"] = universe.groupby("permno")["mktcap"].shift(1)
ok = universe.dropna(subset=["ret", "mktcap_lag"]).copy()
ok["w_ret"] = ok["mktcap_lag"] * ok["ret"]

grp = ok.groupby("date")
vw = grp["w_ret"].sum() / grp["mktcap_lag"].sum()

print(f"months: {len(vw)} (first month has no lag and drops out)")
print(ann_stats(vw, "VW"))
print(f"growth of £1: £{(1 + vw).prod():.2f}")

For the example universe that is 119 months, an annualised mean around 17.7%, a Sharpe ratio around 1.15, and £1 growing to roughly £5.08 — this tech-and-banks universe is mega-cap-heavy, so VW beats EW here even though EW wins on the full market (Notes 04 §2.6 explains why both can be true).

Now the two traps. **Same-month caps** weight month-$t$ returns by a cap that already contains the month-$t$ return: winners get mechanically overweighted *before the fact*. That is look-ahead bias, and it inflates measured performance — section 9 quantifies by how much.

**A bare `shift(1)`** ignores permno boundaries: each stock's first row inherits the *previous stock's last* market cap. In this balanced panel that corrupts one month per stock; in a real universe, where histories have gaps and deaths, it leaks mid-sample and is much harder to spot. Always lag inside `groupby("permno")`.

### 6. MVO where it works

MVO behaves itself when the universe is small relative to the sample. Take the five largest stocks by market cap in the final month, compute their closed-form weights at $\gamma = 10$, and check the condition number of their covariance matrix on the way past:


In [ ]:
last = universe[universe["date"] == universe["date"].max()]
top5 = last.nlargest(5, "mktcap")["permno"].tolist()
print(f"five largest permnos: {top5}")

five = rets[top5]
print(f"condition number: {np.linalg.cond(five.cov().to_numpy()):.1f}")
pd.Series(mvo_weights(five), index=top5, name="weight")

For the example universe the five largest are permnos 14593, 10107, 84788, 90319 and 91233 — Apple, Microsoft, Amazon, Alphabet and Mastercard — with a condition number around 7 (an entirely healthy matrix: at $N = 5$, $T = 120$, the optimiser is in its comfort zone) and weights of roughly 0.22, 0.18, 0.14, **−0.07** and 0.53.

The negative weight is a **short position**: unconstrained MVO shorts freely. Alphabet is shorted not because the optimiser expects it to fall, but because it is highly correlated with the other tech names while carrying a lower in-sample mean — shorting it partially hedges the others, freeing risk budget for Mastercard's high mean and comparatively low correlation. That is also why the largest in-sample mean does not simply collect the largest weight: MVO weighs means *against* covariances, so weight ordering need not follow mean ordering. Whether these fine-tuned in-sample trade-offs survive out of sample is section 8.

### 7. Break it, then explain the wreckage

Now the same function on the *entire* universe. Before running it, it is worth predicting what happens — hold that prediction, because the point of this section is that the failure is arithmetic rather than bad luck. Then we dissect the wreckage: the condition number and rank of $\Sigma$, the number of parameters we have asked 120 months of data to estimate, and the two reasons the answer was never going to be usable.


In [ ]:
Sigma_all = rets.cov().to_numpy()
T, N = rets.shape
print(f"N = {N}, T = {T}")
print(f"rank(Sigma) = {np.linalg.matrix_rank(Sigma_all)}")
print(f"condition number = {np.linalg.cond(Sigma_all):.2e}")

try:
    w_all = mvo_weights(rets)
    print(f"largest |weight|: {np.abs(w_all).max():,.0f}")
except np.linalg.LinAlgError as e:
    print(f"LinAlgError: {e}")
except AssertionError:
    print("AssertionError: weights do not sum to one — solve() returned garbage")

For the example universe (452 stocks) that is rank 119 and a condition number of order $10^{20}$, followed by one of three endings — all equally damning:

- `LinAlgError: Singular matrix` — NumPy refuses outright;
- an `AssertionError` — `solve` returned *something*, but the weights no longer sum to one (in one run on the current snapshot they summed to −4, with individual "weights" around $10^{15}$; the exact garbage depends on your machine's linear-algebra library), and the Week 02 assertion caught it;
- no error but absurd weights — the worst outcome, because nothing warned you.

**Parameter count:** $\hat\mu$ and $\hat\Sigma$ together need $N + \tfrac{N^2+N}{2}$ numbers. At $N = 452$ that is $452 + 102{,}378 = 102{,}830$ parameters from $120 \times 452 = 54{,}240$ data points — about **1.9 parameters per data point**.

**The two reasons, one sentence each.** *Mathematics:* a covariance matrix estimated from $T = 120$ months has rank at most $T - 1 = 119$ (Notes 04 §4.2), so at $N = 452$ it is singular by construction — before any discussion of noise. *Statistics:* even if it were invertible, estimating two parameters per observation guarantees the estimates are mostly noise, and $\Sigma^{-1}$ amplifies exactly the worst-estimated directions — Michaud's "error maximiser". The fixes (single-index structure, shrinkage, constraints) are in Notes 04 §5; Week 05 supplies the factor they rely on.

### 8. The honest race

Time for the honest test: the lecture's rolling backtest on the five stocks from section 6. At each month $t$ we estimate on months $t-60$ to $t-1$, hold for month $t$, and charge 10bps on two-way turnover $\text{TO}_t = \sum_i |w_{i,t} - w_{i,t-1}|$. The scoreboard we want is gross Sharpe, net Sharpe and average turnover, for MVO against 1/N:


In [ ]:
WINDOW, GAMMA, COST = 60, 10.0, 0.0010
n = five.shape[1]
results, prev = [], {"1/N": None, "MVO": None}

for t in range(WINDOW, len(five)):
    train = five.iloc[t - WINDOW:t]          # strictly before month t: no peeking
    weights = {"1/N": np.full(n, 1 / n), "MVO": mvo_weights(train, GAMMA)}
    for name, wt in weights.items():
        gross = float(wt @ five.iloc[t])
        to = 1.0 if prev[name] is None else float(np.abs(wt - prev[name]).sum())
        results.append({"date": five.index[t], "strategy": name,
                        "gross": gross, "turnover": to,
                        "net": gross - COST * to})
        prev[name] = wt

bt = pd.DataFrame(results)

def ann_sharpe(x):
    return np.sqrt(12) * x.mean() / x.std()

bt.groupby("strategy").agg(sharpe_gross=("gross", ann_sharpe),
                           sharpe_net=("net", ann_sharpe),
                           avg_turnover=("turnover", "mean"))

For the example five the scoreboard reads roughly: 1/N net Sharpe ≈ 1.58 against MVO ≈ 1.46, with average monthly turnover of about 0.017 for 1/N (entirely the initial purchase) against about 0.19 for MVO — the optimiser trades **more than ten times** as much, because every new estimation window lurches its weights. In this sample 1/N wins even *gross*; costs only widen the gap.

The obvious next question is whether the ranking survives a heavier cost rate. At 20bps:

In [ ]:
bt["net20"] = bt["gross"] - 0.0020 * bt["turnover"]
bt.groupby("strategy")["net20"].apply(ann_sharpe)

The ranking is unchanged (≈ 1.58 vs ≈ 1.45): doubling the cost rate barely touches 1/N and taxes MVO further. The decisive number was always **average turnover** — the cost rate only scales a gap that turnover created. Two honest caveats belong with any write-up of a race like this: 120 months minus a 60-month training window leaves only 60 out-of-sample months (a short sample — Week 07 returns to this), and in other samples MVO does win gross and only loses net; here it manages neither. Either way, DeMiguel et al. (2009) would not be surprised.

### 9. Verify the AI

The last piece of the kit is the habit that keeps the rest of it honest. Asked for a value-weighted portfolio of the universe, an AI assistant produced the code below, with this explanation attached:

*"Here is a clean, fully vectorised value-weighted portfolio. I pivot the market caps into the same date-by-permno layout as `rets`, normalise them row-wise so the weights sum to exactly 1.0 in every month — which I have verified — and take the weighted sum:"*

In [ ]:
caps = universe.pivot(index="date", columns="permno", values="mktcap")

w = caps.div(caps.sum(axis=1), axis=0)   # each row sums to exactly 1.0
vw_ai = (w * rets).sum(axis=1)

The code runs without error, the weights genuinely do sum to one, and the output looks entirely plausible. So we go looking: find the flaw, quantify what it does to the annualised mean against the honest series from section 5, and fix it in one line.


**The weights are built from month-$t$ market caps and applied to month-$t$ returns** — same-month weighting, i.e. look-ahead bias. It is the lecture's AI-checkpoint bug in a different costume: there is no `shift` to inspect and no `groupby` to check, so nothing *looks* missing, and the assistant's confident (and true) claim that "each row sums to exactly 1.0" verifies the property that was never in doubt while saying nothing about the timing that was. Everything the AI said is correct; the code is still wrong.

Quantifying the damage:

In [ ]:
common = vw.index                        # the honest lagged series from section 5
print(f"AI (look-ahead) annualised mean: {12 * vw_ai.loc[common].mean():.2%}")
print(f"honest (lagged) annualised mean: {12 * vw.mean():.2%}")

For the example universe that is roughly 21.4% against 17.7% — **almost 4 percentage points a year of pure time travel**. If a backtest ever looks that much better after an innocent "refactor", this bug is the first suspect.

The one-line fix — lag the weight matrix, so month-$t$ returns meet end-of-month-$(t-1)$ weights:

In [ ]:
w_fixed = w.shift(1)                     # rows still sum to one; first month drops
vw_fixed = (w_fixed * rets).sum(axis=1).iloc[1:]
print(f"max |fixed - lagged VW| = {(vw_fixed - vw).abs().max():.2e}")

The two series agree at machine precision ($\sim 10^{-16}$). Note that `shift(1)` on the weight table is safe *here* because each column is one permno — the leak-across-stocks danger described in section 5 arises only when the stocks are stacked in one column, as in `universe`. Written up as evidence → diagnosis → fix in under 150 words, this is exactly the shape a bug report should take.

## Before you leave
#### ⚠️ Common mistakes this week
- **Lagging without `groupby("permno")`** — the lag leaks across stocks. In a table with one row per stock-month, always `groupby("permno")[...].shift(1)`.
- **`dropna()` without `subset=`** — one missing value in any column deletes the row; on a universe with a delisted stock this can silently erase most of your panel. Name the columns: `dropna(subset=["ret", "mktcap_lag"])`.
- **Same-month weights** — if your backtest suddenly looks brilliant, check the timing before celebrating. Weights must be computable from information available *before* the return they multiply.
- **`np.linalg.inv(Sigma) @ mu` instead of `solve`** — works at $N = 5$, but it is slower, less stable, and teaches the wrong reflex for the matrices where it matters.
- **Annualisation slips** — Sharpe scales with $\sqrt{12}$, not 12; and if you annualise the mean you must annualise the volatility too.